# Async: `gather` and `amap`

Concurrent asyncio work, such as many HTTP or LLM API requests, finishes out of order, so a bar wrapped around the inputs counts the wrong thing. psnek has two helpers for it:

1. `psnek.gather` works like `asyncio.gather`, and counts each call as it completes.
2. `psnek.amap` maps an async function over items, with a limit on how many calls run at once.

Both cancel the remaining calls when one fails. Top-level `await` works in Jupyter, so the cells below await directly.


In [1]:
import asyncio
import logging
import time
import tracemalloc
from collections.abc import Awaitable

import psnek

logger = logging.getLogger("demo")


def latency(key: int, low: float, high: float) -> float:
    """Return a repeatable pseudo-random delay between ``low`` and ``high``."""
    return low + (key * 7919 % 1000) / 1000 * (high - low)


async def fetch(page: int) -> int:
    """Pretend to request ``page`` from an API and return its size in bytes."""
    await asyncio.sleep(latency(page, 0.02, 0.2))
    return page * 1024

## The Trap: `track()` Around `gather`

`asyncio.gather(*bar)` creates every coroutine before any of them runs, so the bar reaches 100% immediately.


In [2]:
start = time.perf_counter()
calls = psnek.track([fetch(page) for page in range(300)], "counts creation")
sizes = await asyncio.gather(*calls)
print(
    f"bar finished after {calls.elapsed:.2f}s; "
    f"the requests took {time.perf_counter() - start:.2f}s"
)

# counts creation #######################  100.0%    300/300         0:00   499kit/s  done

bar finished after 0.00s; the requests took 0.20s


## `gather`: Count Completions

`psnek.gather()` takes the same arguments as `asyncio.gather()` and returns results in the same order, but advances the bar as each call completes.


In [3]:
sizes = await psnek.gather(*(fetch(page) for page in range(300)), desc="fetch pages")
print(
    f"fetched {len(sizes)} pages ({sum(sizes) / 2**20:.1f} MiB), in order: {sizes[:3]}"
)

# fetch pages ###########################  100.0%    300/300         0:00   1471it/s  done

fetched 300 pages (43.8 MiB), in order: [0, 1024, 2048]


## `amap`: Limit Concurrency

APIs have rate limits. `psnek.amap(function, items, desc, workers=N)` runs at most `N` calls at once, and returns the results in input order.


In [4]:
in_flight: set[int] = set()
flight_sizes: list[int] = []


async def counted_fetch(page: int) -> int:
    """Fetch ``page`` while recording how many requests are in flight."""
    in_flight.add(page)
    flight_sizes.append(len(in_flight))
    try:
        return await fetch(page)
    finally:
        in_flight.discard(page)


sizes = await psnek.amap(counted_fetch, range(300), "fetch 8 at a time", workers=8)
print(f"fetched {len(sizes)} pages, at most {max(flight_sizes)} requests in flight")

# fetch 8 at a time #####################  100.0%    300/300         0:04   70.2it/s  done

fetched 300 pages, at most 8 requests in flight


### Memory

The usual way to limit `asyncio.gather` is to wrap each call in a semaphore, which still creates a coroutine and a task for every item up front. `amap` only holds `workers` tasks. Peak memory for 100,000 quick calls, 16 at a time:


In [5]:
async def quick(item: int) -> int:
    """Yield to the event loop once, like a fast request."""
    await asyncio.sleep(0)
    return item


async def semaphore_gather(count: int) -> list[int]:
    """Run ``quick`` over ``count`` items, 16 at a time, with a semaphore."""
    limit = asyncio.Semaphore(16)

    async def limited(item: int) -> int:
        async with limit:
            return await quick(item)

    calls = (limited(item) for item in range(count))
    return await psnek.gather(*calls, desc="semaphore + gather")


async def peak_mib(work: Awaitable[object]) -> float:
    """Return the peak traced memory, in MiB, while awaiting ``work``."""
    tracemalloc.start()
    try:
        await work
        return tracemalloc.get_traced_memory()[1] / 2**20
    finally:
        tracemalloc.stop()


gathered = await peak_mib(semaphore_gather(100_000))
mapped = await peak_mib(psnek.amap(quick, range(100_000), "amap", workers=16))
print(f"semaphore + gather: {gathered:5.1f} MiB peak")
print(f"amap, 16 workers:   {mapped:5.1f} MiB peak")

# semaphore + gather ################  100.0%  100000/100000         0:02  39.8kit/s  done
# amap               ################  100.0%  100000/100000         0:00   277kit/s  done

semaphore + gather: 167.6 MiB peak
amap, 16 workers:     4.2 MiB peak


## A Failure Cancels the Rest

When one call raises, `asyncio.gather` raises its exception but leaves the other calls running in the background. `psnek.gather` and `amap` cancel them first, then raise the original exception and mark the bar as failed.


In [6]:
running: set[int] = set()


async def flaky(page: int) -> int:
    """Take a second to fetch ``page``, except page 13, which fails fast."""
    running.add(page)
    try:
        if page == 13:
            await asyncio.sleep(0.05)
            raise ConnectionError(f"page {page}: connection reset")
        await asyncio.sleep(1)
        return page * 1024
    finally:
        running.discard(page)


try:
    await asyncio.gather(*(flaky(page) for page in range(50)))
except ConnectionError as error:
    logger.error("asyncio.gather: %s; %d calls still running", error, len(running))
await asyncio.sleep(1.1)  # let the calls asyncio.gather left behind finish

try:
    await psnek.gather(*(flaky(page) for page in range(50)), desc="flaky pages")
except ConnectionError as error:
    logger.error("psnek.gather: %s; %d calls still running", error, len(running))

asyncio.gather: page 13: connection reset; 49 calls still running


x flaky pages -__________________    2.0%       1/50         0:00   18.4it/s  failed ConnectionError

psnek.gather: page 13: connection reset; 0 calls still running


### Keep Going with `return_exceptions`

When some failures are expected, `return_exceptions=True` returns each exception in place of its result, as with `asyncio.gather`.


In [7]:
results = await psnek.gather(
    *(flaky(page) for page in range(50)),
    desc="flaky pages, keep going",
    return_exceptions=True,
)
failures = [result for result in results if isinstance(result, BaseException)]
print(f"{len(results) - len(failures)} pages fetched, failures: {failures}")

# flaky pages, keep going ###############  100.0%      50/50         0:01   49.8it/s  done

49 pages fetched, failures: [ConnectionError('page 13: connection reset')]


## Bars Inside Each Call

A bar started inside a call is drawn below the top-level bar, beside the bars of the other calls. Here, three workers each download shards with a bar of their own.


In [8]:
CHUNK = 64 * 1024
CHUNKS = 20


async def download(shard: int) -> int:
    """Download ``shard`` in chunks, with a bar counting its bytes."""
    options = psnek.Options(unit="B", scale="iec")
    name = f"shard-{shard:02}.parquet"
    with psnek.Bar(CHUNKS * CHUNK, name, options=options) as bar:
        for chunk in range(CHUNKS):
            await asyncio.sleep(latency(shard * CHUNKS + chunk, 0.01, 0.05))
            bar.update(CHUNK)
    return CHUNKS * CHUNK


sizes = await psnek.amap(download, range(12), "download shards", workers=3)
print(f"downloaded {sum(sizes) / 2**20:.1f} MiB")

# download shards    ##############  100.0%            12/12         0:02    4.6it/s  done

downloaded 15.0 MiB
